# Celebal Technologies – Week 6 Assignment

## Spark Architecture and Performance Optimization using PySpark

### Submitted By

**Amit Singh**


# Creating Spark Session

The Spark Session is the entry point of every Spark application.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark = SparkSession.builder \
    .appName("Celebal Week 6 Assignment") \
    .getOrCreate()

In [0]:

df = spark.read.csv(
    "/Volumes/workspace/default/new_data/Ecommerce_Sales_Data_2024_2025.csv",
    header=True,
    inferSchema=True
)

In [0]:
df.show()

+--------+----------+-------------------+------+-----------+-----------+------------+--------------------+--------+----------+--------+---------+--------+------------+
|Order ID|Order Date|      Customer Name|Region|       City|   Category|Sub-Category|        Product Name|Quantity|Unit Price|Discount|    Sales|  Profit|Payment Mode|
+--------+----------+-------------------+------+-----------+-----------+------------+--------------------+--------+----------+--------+---------+--------+------------+
|   10001|2024-10-19|       Kashvi Varty| South|  Bangalore|      Books| Non-Fiction|   Non-Fiction Ipsum|       2|     36294|       5|  68958.6|10525.09|  Debit Card|
|   10002|2025-08-30|        Advik Desai| North|      Delhi|  Groceries|        Rice|           Rice Nemo|       1|     42165|      20|  33732.0| 6299.66|  Debit Card|
|   10003|2023-11-04|         Rhea Kalla|  East|      Patna|    Kitchen|      Juicer|         Juicer Odio|       4|     64876|      20| 207603.2|19850.27| Credi

# Demonstrating Lazy Evaluation


In [0]:
lazy_df = (
    df.filter(col("Category") == "Electronics")
    .select(
        "Order ID",
        "Customer Name",
        "Product Name",
        "Sales"
    )
    .withColumnRenamed(
        "Sales",
        "Total Sales"
    )
)

At this stage, Spark has only prepared the logical execution plan.

No computation has occurred because no Action has been called.

In [0]:
lazy_df.show(5)

+--------+--------------------+-----------------+-----------+
|Order ID|       Customer Name|     Product Name|Total Sales|
+--------+--------------------+-----------------+-----------+
|   10016|     Stuvan Majumdar|   Laptop Numquam|   232402.5|
|   10017|   Ritvik Ramanathan|    Smartwatch Ad|  134362.05|
|   10025|Lakshay Ramakrishnan|Headphones Itaque|    96224.0|
|   10043|          Badal Dada|         Camera A|   137260.0|
|   10078|         Chirag Hans|   Camera Nostrum|    53998.4|
+--------+--------------------+-----------------+-----------+
only showing top 5 rows


The `show()` function is an Action.


In [0]:
print("Total Records :", lazy_df.count())

Total Records : 472


##CSV vs Parquet

In [0]:

df = spark.read.csv(
    "/Volumes/workspace/default/new_data/Ecommerce_Sales_Data_2024_2025.csv",
    header=True,
    inferSchema=True
)

Reading data in csv

In [0]:
df.show(5)

+--------+----------+-------------+------+---------+---------+------------+-----------------+--------+----------+--------+--------+--------+------------+
|Order ID|Order Date|Customer Name|Region|     City| Category|Sub-Category|     Product Name|Quantity|Unit Price|Discount|   Sales|  Profit|Payment Mode|
+--------+----------+-------------+------+---------+---------+------------+-----------------+--------+----------+--------+--------+--------+------------+
|   10001|2024-10-19| Kashvi Varty| South|Bangalore|    Books| Non-Fiction|Non-Fiction Ipsum|       2|     36294|       5| 68958.6|10525.09|  Debit Card|
|   10002|2025-08-30|  Advik Desai| North|    Delhi|Groceries|        Rice|        Rice Nemo|       1|     42165|      20| 33732.0| 6299.66|  Debit Card|
|   10003|2023-11-04|   Rhea Kalla|  East|    Patna|  Kitchen|      Juicer|      Juicer Odio|       4|     64876|      20|207603.2|19850.27| Credit Card|
|   10004|2025-05-23|    Anika Sen|  East|  Kolkata|Groceries|         Oil| 

In [0]:
df.printSchema()

root
 |-- Order ID: integer (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit Price: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Payment Mode: string (nullable = true)



In [0]:
df.columns

['Order ID',
 'Order Date',
 'Customer Name',
 'Region',
 'City',
 'Category',
 'Sub-Category',
 'Product Name',
 'Quantity',
 'Unit Price',
 'Discount',
 'Sales',
 'Profit',
 'Payment Mode']

In [0]:
print("Total Records :", df.count())

Total Records : 5000


In [0]:
print("Total Columns :", len(df.columns))

Total Columns : 14


In [0]:
df.dtypes

[('Order ID', 'int'),
 ('Order Date', 'date'),
 ('Customer Name', 'string'),
 ('Region', 'string'),
 ('City', 'string'),
 ('Category', 'string'),
 ('Sub-Category', 'string'),
 ('Product Name', 'string'),
 ('Quantity', 'int'),
 ('Unit Price', 'int'),
 ('Discount', 'int'),
 ('Sales', 'double'),
 ('Profit', 'double'),
 ('Payment Mode', 'string')]

In [0]:
df.describe().show()

+-------+------------------+--------------------+------+------------------+--------+------------+-------------------+------------------+------------------+-----------------+------------------+------------------+------------+
|summary|          Order ID|       Customer Name|Region|              City|Category|Sub-Category|       Product Name|          Quantity|        Unit Price|         Discount|             Sales|            Profit|Payment Mode|
+-------+------------------+--------------------+------+------------------+--------+------------+-------------------+------------------+------------------+-----------------+------------------+------------------+------------+
|  count|              5000|                5000|  5000|              5000|    5000|        5000|               5000|              5000|              5000|             5000|              5000|              5000|        5000|
|   mean|           12500.5|                NULL|  NULL|              NULL|    NULL|        NULL|   

# Parquet

Parquet is a columnar storage format designed for big data processing.

In [0]:
df.write.mode("overwrite").parquet(
"/Volumes/workspace/default/new_data/ecommerce_parquet"
)

In [0]:
parquet_df = spark.read.parquet(
"/Volumes/workspace/default/new_data/ecommerce_parquet"
)

In [0]:
parquet_df.show(5)

+--------+----------+-------------+------+---------+---------+------------+-----------------+--------+----------+--------+--------+--------+------------+
|Order ID|Order Date|Customer Name|Region|     City| Category|Sub-Category|     Product Name|Quantity|Unit Price|Discount|   Sales|  Profit|Payment Mode|
+--------+----------+-------------+------+---------+---------+------------+-----------------+--------+----------+--------+--------+--------+------------+
|   10001|2024-10-19| Kashvi Varty| South|Bangalore|    Books| Non-Fiction|Non-Fiction Ipsum|       2|     36294|       5| 68958.6|10525.09|  Debit Card|
|   10002|2025-08-30|  Advik Desai| North|    Delhi|Groceries|        Rice|        Rice Nemo|       1|     42165|      20| 33732.0| 6299.66|  Debit Card|
|   10003|2023-11-04|   Rhea Kalla|  East|    Patna|  Kitchen|      Juicer|      Juicer Odio|       4|     64876|      20|207603.2|19850.27| Credit Card|
|   10004|2025-05-23|    Anika Sen|  East|  Kolkata|Groceries|         Oil| 

In [0]:
parquet_df.printSchema()

root
 |-- Order ID: integer (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit Price: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Payment Mode: string (nullable = true)



In [0]:
parquet_df.select(
"Customer Name",
"Sales"
).show()

+-------------------+---------+
|      Customer Name|    Sales|
+-------------------+---------+
|       Kashvi Varty|  68958.6|
|        Advik Desai|  33732.0|
|         Rhea Kalla| 207603.2|
|          Anika Sen| 158610.0|
|        Akarsh Kaul|  45033.3|
|Vardaniya Jayaraman|171219.75|
|      Drishya Khare|   6908.4|
|          Misha Dua|  14296.5|
|        Arhaan Vala|  32667.2|
|      Lavanya Hayer|  28366.2|
|    Divyansh Thaman|  17217.6|
|      Nishith Kumar|  73078.2|
|       Anika Khanna| 51253.45|
|      Amani Acharya| 119215.0|
|        Anay Grewal|  29343.6|
|    Stuvan Majumdar| 232402.5|
|  Ritvik Ramanathan|134362.05|
|     Darshit Sharma|  13590.0|
|        Sara Chanda|  39789.6|
|         Gatik Ravi|172866.75|
+-------------------+---------+
only showing top 20 rows


### Selecting Specific Columns

In [0]:
selected_df = df.select(
    "Order ID",
    "Customer Name",
    "Category",
    "Sales",
    "Profit"
)
selected_df.show(5)

+--------+-------------+---------+--------+--------+
|Order ID|Customer Name| Category|   Sales|  Profit|
+--------+-------------+---------+--------+--------+
|   10001| Kashvi Varty|    Books| 68958.6|10525.09|
|   10002|  Advik Desai|Groceries| 33732.0| 6299.66|
|   10003|   Rhea Kalla|  Kitchen|207603.2|19850.27|
|   10004|    Anika Sen|Groceries|158610.0|36311.02|
|   10005|  Akarsh Kaul| Clothing| 45033.3| 9050.04|
+--------+-------------+---------+--------+--------+
only showing top 5 rows


In [0]:
electronics_df = df.filter(
    col("Category") == "Electronics"
).select(
    "Order ID",
    "Product Name",
    "Sales"
)
electronics_df.show(10)

+--------+------------------+---------+
|Order ID|      Product Name|    Sales|
+--------+------------------+---------+
|   10016|    Laptop Numquam| 232402.5|
|   10017|     Smartwatch Ad|134362.05|
|   10025| Headphones Itaque|  96224.0|
|   10043|          Camera A| 137260.0|
|   10078|    Camera Nostrum|  53998.4|
|   10092|       Camera Ipsa|  62529.4|
|   10095| Camera Temporibus|  65308.8|
|   10100|      Mobile Harum| 201636.0|
|   10111|Camera Dignissimos|  54538.2|
|   10122|       Laptop Amet|   6284.0|
+--------+------------------+---------+
only showing top 10 rows


In [0]:
north_df = df.filter(
    col("Region") == "North"
)
north_df.show(10)

+--------+----------+--------------------+------+----------+-----------+------------+--------------------+--------+----------+--------+--------+--------+------------+
|Order ID|Order Date|       Customer Name|Region|      City|   Category|Sub-Category|        Product Name|Quantity|Unit Price|Discount|   Sales|  Profit|Payment Mode|
+--------+----------+--------------------+------+----------+-----------+------------+--------------------+--------+----------+--------+--------+--------+------------+
|   10002|2025-08-30|         Advik Desai| North|     Delhi|  Groceries|        Rice|           Rice Nemo|       1|     42165|      20| 33732.0| 6299.66|  Debit Card|
|   10008|2025-08-17|           Misha Dua| North|   Lucknow|      Books|   Biography|       Biography Vel|       1|     15885|      10| 14296.5| 1289.03|  Debit Card|
|   10009|2025-03-07|         Arhaan Vala| North|    Jaipur|  Groceries|      Spices|     Spices Expedita|       1|     40834|      20| 32667.2| 3700.89| Credit Card

In [0]:
high_sales = df.filter(
    (col("Region") == "North") &
    (col("Sales") > 1000)
)
high_sales.show()

+--------+----------+--------------------+------+----------+-----------+------------+--------------------+--------+----------+--------+---------+--------+------------+
|Order ID|Order Date|       Customer Name|Region|      City|   Category|Sub-Category|        Product Name|Quantity|Unit Price|Discount|    Sales|  Profit|Payment Mode|
+--------+----------+--------------------+------+----------+-----------+------------+--------------------+--------+----------+--------+---------+--------+------------+
|   10002|2025-08-30|         Advik Desai| North|     Delhi|  Groceries|        Rice|           Rice Nemo|       1|     42165|      20|  33732.0| 6299.66|  Debit Card|
|   10008|2025-08-17|           Misha Dua| North|   Lucknow|      Books|   Biography|       Biography Vel|       1|     15885|      10|  14296.5| 1289.03|  Debit Card|
|   10009|2025-03-07|         Arhaan Vala| North|    Jaipur|  Groceries|      Spices|     Spices Expedita|       1|     40834|      20|  32667.2| 3700.89| Credi

In [0]:
filtered_df = df.filter(
    (col("Region") == "North") |
    (col("Category") == "Electronics")
)
filtered_df.show()

+--------+----------+--------------------+------+-----------+-----------+------------+--------------------+--------+----------+--------+---------+--------+------------+
|Order ID|Order Date|       Customer Name|Region|       City|   Category|Sub-Category|        Product Name|Quantity|Unit Price|Discount|    Sales|  Profit|Payment Mode|
+--------+----------+--------------------+------+-----------+-----------+------------+--------------------+--------+----------+--------+---------+--------+------------+
|   10002|2025-08-30|         Advik Desai| North|      Delhi|  Groceries|        Rice|           Rice Nemo|       1|     42165|      20|  33732.0| 6299.66|  Debit Card|
|   10008|2025-08-17|           Misha Dua| North|    Lucknow|      Books|   Biography|       Biography Vel|       1|     15885|      10|  14296.5| 1289.03|  Debit Card|
|   10009|2025-03-07|         Arhaan Vala| North|     Jaipur|  Groceries|      Spices|     Spices Expedita|       1|     40834|      20|  32667.2| 3700.89|

### Renaming Columns

In [0]:
df_transform = df.withColumnRenamed(
    "Sales",
    "Total Sales"
)

In [0]:
df_transform = df_transform.withColumnRenamed(
    "Profit",
    "Net Profit"
)

In [0]:
df_transform.printSchema()

root
 |-- Order ID: integer (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit Price: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Total Sales: double (nullable = true)
 |-- Net Profit: double (nullable = true)
 |-- Payment Mode: string (nullable = true)



### Casting Data Types

In [0]:
df_transform = df_transform.withColumn(
    "Quantity",
    col("Quantity").cast("Integer")
)

In [0]:
df_transform.printSchema()

root
 |-- Order ID: integer (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit Price: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Total Sales: double (nullable = true)
 |-- Net Profit: double (nullable = true)
 |-- Payment Mode: string (nullable = true)



### Creating a New Column

In [0]:
df_transform = df_transform.withColumn(
    "Sales with GST",
    col("Total Sales") * 1.18
)

In [0]:
df_transform = df_transform.withColumn(
    "Profit Margin",
    round(
        (col("Net Profit") / col("Total Sales")) * 100,
        2
    )
)

In [0]:
df_transform.select(
    "Total Sales",
    "Net Profit",
    "Profit Margin"
).show(10)

+-----------+----------+-------------+
|Total Sales|Net Profit|Profit Margin|
+-----------+----------+-------------+
|    68958.6|  10525.09|        15.26|
|    33732.0|   6299.66|        18.68|
|   207603.2|  19850.27|         9.56|
|   158610.0|  36311.02|        22.89|
|    45033.3|   9050.04|         20.1|
|  171219.75|  23722.84|        13.86|
|     6908.4|    680.26|         9.85|
|    14296.5|   1289.03|         9.02|
|    32667.2|   3700.89|        11.33|
|    28366.2|   5703.09|        20.11|
+-----------+----------+-------------+
only showing top 10 rows


In [0]:
df_transform.select(
    "Order ID",
    "Customer Name",
    "Product Name",
    "Region",
    "Category",
    "Total Sales",
    "Net Profit",
    "Profit Margin"
).show(10)

+--------+-------------------+-------------------+------+---------+-----------+----------+-------------+
|Order ID|      Customer Name|       Product Name|Region| Category|Total Sales|Net Profit|Profit Margin|
+--------+-------------------+-------------------+------+---------+-----------+----------+-------------+
|   10001|       Kashvi Varty|  Non-Fiction Ipsum| South|    Books|    68958.6|  10525.09|        15.26|
|   10002|        Advik Desai|          Rice Nemo| North|Groceries|    33732.0|   6299.66|        18.68|
|   10003|         Rhea Kalla|        Juicer Odio|  East|  Kitchen|   207603.2|  19850.27|         9.56|
|   10004|          Anika Sen|      Oil Doloribus|  East|Groceries|   158610.0|  36311.02|        22.89|
|   10005|        Akarsh Kaul|      Kids Wear Quo|  West| Clothing|    45033.3|   9050.04|         20.1|
|   10006|Vardaniya Jayaraman|    Chair Assumenda|  West|Furniture|  171219.75|  23722.84|        13.86|
|   10007|      Drishya Khare| Accessories Minima|  Wes

### Sorting the Dataset

In [0]:
df_transform.orderBy(
    col("Total Sales").desc()
).show(10)

+--------+----------+--------------+------+------------------+-----------+-------------+--------------------+--------+----------+--------+-----------+----------+------------+------------------+-------------+
|Order ID|Order Date| Customer Name|Region|              City|   Category| Sub-Category|        Product Name|Quantity|Unit Price|Discount|Total Sales|Net Profit|Payment Mode|    Sales with GST|Profit Margin|
+--------+----------+--------------+------+------------------+-----------+-------------+--------------------+--------+----------+--------+-----------+----------+------------+------------------+-------------+
|   13333|2025-02-07|   Rohan Khare|  West|             Surat|     Sports|    Dumbbells|      Dumbbells Fuga|       5|     79697|       0|   398485.0|  21266.93|         COD|          470212.3|         5.34|
|   13330|2024-10-01|  Rasha Saxena| South|        Coimbatore| Home Decor|         Lamp|         Lamp Libero|       5|     78716|       0|   393580.0|  75345.21| Net Ba

# Performance Optimization in Apache Spark

In [0]:
sales_by_region = df_transform.groupBy(
    "Region"
).agg(
    sum("Total Sales").alias("Regional Sales")
)
sales_by_region.show()

+------+--------------------+
|Region|      Regional Sales|
+------+--------------------+
| North|1.4357824610000005E8|
|  East|1.3581163794999996E8|
| South|1.2323016695000012E8|
|  West|1.3104597334999998E8|
+------+--------------------+



In [0]:
sales_by_region.orderBy(
    col("Regional Sales").desc()
).show()

+------+--------------------+
|Region|      Regional Sales|
+------+--------------------+
| North|1.4357824610000005E8|
|  East|1.3581163794999996E8|
|  West|1.3104597334999998E8|
| South|1.2323016695000012E8|
+------+--------------------+



In [0]:
parquet_df.filter(
    col("Region") == "North"
).show()

+--------+----------+--------------------+------+----------+-----------+------------+--------------------+--------+----------+--------+---------+--------+------------+
|Order ID|Order Date|       Customer Name|Region|      City|   Category|Sub-Category|        Product Name|Quantity|Unit Price|Discount|    Sales|  Profit|Payment Mode|
+--------+----------+--------------------+------+----------+-----------+------------+--------------------+--------+----------+--------+---------+--------+------------+
|   10002|2025-08-30|         Advik Desai| North|     Delhi|  Groceries|        Rice|           Rice Nemo|       1|     42165|      20|  33732.0| 6299.66|  Debit Card|
|   10008|2025-08-17|           Misha Dua| North|   Lucknow|      Books|   Biography|       Biography Vel|       1|     15885|      10|  14296.5| 1289.03|  Debit Card|
|   10009|2025-03-07|         Arhaan Vala| North|    Jaipur|  Groceries|      Spices|     Spices Expedita|       1|     40834|      20|  32667.2| 3700.89| Credi

In [0]:
from pyspark.sql.functions import when, count
df_transform.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df_transform.columns
]).show()

+--------+----------+-------------+------+----+--------+------------+------------+--------+----------+--------+-----------+----------+------------+--------------+-------------+
|Order ID|Order Date|Customer Name|Region|City|Category|Sub-Category|Product Name|Quantity|Unit Price|Discount|Total Sales|Net Profit|Payment Mode|Sales with GST|Profit Margin|
+--------+----------+-------------+------+----+--------+------------+------------+--------+----------+--------+-----------+----------+------------+--------------+-------------+
|       0|         0|            0|     0|   0|       0|           0|           0|       0|         0|       0|          0|         0|           0|             0|            0|
+--------+----------+-------------+------+----+--------+------------+------------+--------+----------+--------+-----------+----------+------------+--------------+-------------+



In [0]:
efficient_df = df_transform.filter(
    col("Sales") > 500
)
efficient_df.show(10)

+--------+----------+-------------------+------+---------+---------+------------+-------------------+--------+----------+--------+-----------+----------+------------+------------------+-------------+
|Order ID|Order Date|      Customer Name|Region|     City| Category|Sub-Category|       Product Name|Quantity|Unit Price|Discount|Total Sales|Net Profit|Payment Mode|    Sales with GST|Profit Margin|
+--------+----------+-------------------+------+---------+---------+------------+-------------------+--------+----------+--------+-----------+----------+------------+------------------+-------------+
|   10001|2024-10-19|       Kashvi Varty| South|Bangalore|    Books| Non-Fiction|  Non-Fiction Ipsum|       2|     36294|       5|    68958.6|  10525.09|  Debit Card|         81371.148|        15.26|
|   10002|2025-08-30|        Advik Desai| North|    Delhi|Groceries|        Rice|          Rice Nemo|       1|     42165|      20|    33732.0|   6299.66|  Debit Card|39803.759999999995|        18.68|


In [0]:
df_transform.select(
    "Customer Name",
    "Product Name",

).show()

+-------------------+--------------------+
|      Customer Name|        Product Name|
+-------------------+--------------------+
|       Kashvi Varty|   Non-Fiction Ipsum|
|        Advik Desai|           Rice Nemo|
|         Rhea Kalla|         Juicer Odio|
|          Anika Sen|       Oil Doloribus|
|        Akarsh Kaul|       Kids Wear Quo|
|Vardaniya Jayaraman|     Chair Assumenda|
|      Drishya Khare|  Accessories Minima|
|          Misha Dua|       Biography Vel|
|        Arhaan Vala|     Spices Expedita|
|      Lavanya Hayer| Juicer Voluptatibus|
|    Divyansh Thaman|Cookware Set Dolo...|
|      Nishith Kumar|      Perfume Itaque|
|       Anika Khanna|        Vase Ratione|
|      Amani Acharya|  Women's Wear Optio|
|        Anay Grewal|      Juicer Debitis|
|    Stuvan Majumdar|      Laptop Numquam|
|  Ritvik Ramanathan|       Smartwatch Ad|
|     Darshit Sharma|            Oil Odio|
|        Sara Chanda|    Dumbbells Soluta|
|         Gatik Ravi|  Textbook Inventore|
+----------

In [0]:
df_transform.filter(
    col("Total Sales") > 500
).select(
    "Customer Name",
    "Total Sales"
).explain()

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [Customer Name#15643, Sales#15652 AS Total Sales#15657]
      +- PhotonFilter (isnotnull(Sales#15652) AND (Sales#15652 > 500.0))
         +- PhotonRowToColumnar
            +- FileScan csv [Customer Name#15643,Sales#15652] Batched: false, DataFilters: [isnotnull(Sales#15652), (Sales#15652 > 500.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/new_data/Ecommerce_Sales_Data_2024_202..., PartitionFilters: [], PushedFilters: [IsNotNull(Sales), GreaterThan(Sales,500.0)], ReadSchema: struct<Customer Name:string,Sales:double>


== Photon Explanation ==
The query is fully supported by Photon.


# Building an End-to-End Spark Data Pipeline

In [0]:
pipeline_df = spark.read.csv(
    "/Volumes/workspace/default/new_data/Ecommerce_Sales_Data_2024_2025.csv",
    header=True,
    inferSchema=True
)

In [0]:
pipeline_df = pipeline_df.fillna({
    "Discount": 0
})

In [0]:
pipeline_df = pipeline_df.withColumnRenamed(
    "Sales",
    "Total Sales"
)
pipeline_df = pipeline_df.withColumnRenamed(
    "Profit",
    "Net Profit"
)

In [0]:
pipeline_df = pipeline_df.withColumn(
    "Quantity",
    col("Quantity").cast("Integer")
)

In [0]:
pipeline_df = pipeline_df.withColumn(
    "Sales with GST",
    col("Total Sales") * 1.18
)

In [0]:
pipeline_df = pipeline_df.filter(
    (col("Region") == "North") &
    (col("Total Sales") > 1000)
)

In [0]:
pipeline_df = pipeline_df.select(
    "Order ID",
    "Customer Name",
    "Region",
    "Category",
    "Product Name",
    "Quantity",
    "Total Sales",
    "Net Profit",
    "Sales with GST"
)

In [0]:
pipeline_df.show(10)

+--------+--------------------+------+-----------+--------------------+--------+-----------+----------+------------------+
|Order ID|       Customer Name|Region|   Category|        Product Name|Quantity|Total Sales|Net Profit|    Sales with GST|
+--------+--------------------+------+-----------+--------------------+--------+-----------+----------+------------------+
|   10002|         Advik Desai| North|  Groceries|           Rice Nemo|       1|    33732.0|   6299.66|39803.759999999995|
|   10008|           Misha Dua| North|      Books|       Biography Vel|       1|    14296.5|   1289.03|          16869.87|
|   10009|         Arhaan Vala| North|  Groceries|     Spices Expedita|       1|    32667.2|   3700.89|         38547.296|
|   10011|     Divyansh Thaman| North|    Kitchen|Cookware Set Dolo...|       2|    17217.6|   4139.82|20316.767999999996|
|   10013|        Anika Khanna| North| Home Decor|        Vase Ratione|       1|   51253.45|   6826.05|60479.070999999996|
|   10015|      

The complete Spark pipeline has successfully loaded, transformed, filtered, and prepared the dataset for further analysis or storage.

In [0]:
pipeline_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/new_data/processed_csv")

In [0]:
pipeline_df.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/default/new_data/processed_parquet")

In [0]:
saved_df = spark.read.parquet(
    "/Volumes/workspace/default/new_data/processed_parquet"
)
saved_df.show(10)

+--------+--------------------+------+-----------+--------------------+--------+-----------+----------+------------------+
|Order ID|       Customer Name|Region|   Category|        Product Name|Quantity|Total Sales|Net Profit|    Sales with GST|
+--------+--------------------+------+-----------+--------------------+--------+-----------+----------+------------------+
|   10002|         Advik Desai| North|  Groceries|           Rice Nemo|       1|    33732.0|   6299.66|39803.759999999995|
|   10008|           Misha Dua| North|      Books|       Biography Vel|       1|    14296.5|   1289.03|          16869.87|
|   10009|         Arhaan Vala| North|  Groceries|     Spices Expedita|       1|    32667.2|   3700.89|         38547.296|
|   10011|     Divyansh Thaman| North|    Kitchen|Cookware Set Dolo...|       2|    17217.6|   4139.82|20316.767999999996|
|   10013|        Anika Khanna| North| Home Decor|        Vase Ratione|       1|   51253.45|   6826.05|60479.070999999996|
|   10015|      